# Consultant Churn Project Jupyter Notebook

## Inserting Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pypq
import pyarrow.csv as pycsv
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn import metrics
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import textwrap

In [41]:
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: mps


## Helper Functions

In [45]:
def read_parquet(path, engine='pyarrow', columns=None, convert_dtypes=True, **args):
    """
    Read a parquet file (or a directory of parquet files) 
    columns: list of columns to read, by default, read all columns
    convert_dtypes: if True, convert datatypes to save RAM (takes extra time)
    """
    name = path.stem 
    column_st = 'columns="all"' if columns is None else f'{columns=!r}'
    print(f'\nReading {column_st} from {path!r} using {engine=!r}.')

    tic = time()
    df = pd.read_parquet(path, engine=engine, columns=columns, **args)
    toc = time()
    print(f'Read {len(df):,} rows from {path.stem!r} in {toc-tic:.2f} sec.')
    
    if convert_dtypes:
        tic = time()
        size_before = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024

        string_cols_d = {}
        for col, dtype in df.dtypes.to_dict().items():
            if dtype == 'object':  # convert object columns to string
                string_cols_d[col] = 'string[python]'
            if col == 'type' or col == 'concept_name':
                if dtype != 'category':
                    string_cols_d[col] = 'category'
            if col == 'publication_month':
                if dtype != 'uint8':
                    string_cols_d[col] = 'uint8'
            if col == 'score':
                if dtype != 'float16':
                    string_cols_d[col] = 'float16'
        # print(f'{string_cols_d=}')
        df = df.astype(string_cols_d) 
        
        size_after = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024
        toc = time()
        print(f'Converting dtypes took {toc-tic:.2f} sec. Size before: {size_before:.2f}GB, after: {size_after:.2f}GB.')
    
    display('Top 3 rows:', df.head(3))
    return df


def peek_parquet(path):
    """
    peeks at a parquet file (or a directory containing parquet files) without reading the whole thing and prints the following:
    * Path
    * schema
    * number of pieces (fragments)
    * number of rows 
    """
    path = Path(path)
    parq_file = pypq.ParquetDataset(path)
    piece_count = len(parq_file.fragments)
    schema = textwrap.indent(parq_file.schema.to_string(), ' '*4)
    row_count = sum(frag.count_rows() for frag in parq_file.fragments)
    if Path(path).is_dir():
      size = sum(Path(frag.path).stat().st_size for frag in parq_file.fragments)
    else:
      size = path.stat().st_size
    
    st = [
        f'Name: {path.stem!r}',  
        f'Path: {str(path)!r}',
        f'Size: {size/1024/1024/1024:.2g} GB',
        f'Files: {piece_count:,}',
        f'Rows: {row_count:,}',
        f'Schema:\n{schema}',
        f'5 random rows:',
    ]
    print('\n'.join(st))
    sample_df = parq_file.fragments[0].head(5).to_pandas()  # read 5 rows from the first fragment
    display(sample_df)

    return

def read_smaller_tables(name):
    """
    Some smaller tables exist as a CSV only
    """
    assert name in ['institutions', 'institutions_geo', 'concepts']
    path = basepath / 'csv-files'/ month / name
    df = pd.read_csv(f'{path}.csv.gz', engine='c')
    return df

def tsv_to_parquet(tsv_path, output_filename=None, dtype=None, **read_csv_kwargs):
    """
    Convert a TSV file to Parquet format and save it in the current directory.
     
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        dtype: Optional dict of column dtypes for reading TSV
        **read_csv_kwargs: Additional arguments to pass to pd.read_csv
    
    Returns:
        Path to the created parquet file
    """

    #get the output path
    notebook_dir = Path.cwd()
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    output_path = notebook_dir / output_filename

    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        df = pd.read_csv(
            tsv_path,
            sep = '\t',
            dtype=dtype,
            low_memory=False, 
            **read_csv_kwargs
        )
        df.to_parquet(output_path, index=False)

        print(f"Successfully wrote Parquet file to: {output_path}")

    return output_path

def csv_to_parquet_pyarrow(csv_path, output_filename=None, column_types=None,drop_columns=None):
    """
    Convert a CSV file to Parquet format using PyArrow and save it in the current directory.
    
    Args:
        csv_path: Path to the CSV file to convert
        output_filename: Optional output filename (defaults to same name as CSV but .parquet)
        drop_columns: Optional list of columns to drop from the CSV before conversion
    Returns:
        Path to the created parquet file
    """

    if output_filename is None:
        output_filename = Path(csv_path).stem + '.parquet'
    
    output_path = Path(output_filename)

    if not output_path.exists():
        print(f"converting {csv_path} to Parquet using PyArrow...")

        parse_options = pycsv.ParseOptions(delimiter=',')
        ##read_options = pycsv.ReadOptions(autogenerate_column_names=True)

        include_columns = None
        if drop_columns:
            peek = pycsv.open_csv(csv_path, parse_options=parse_options)
            all_cols = next(iter(peek)).schema.names
            include_columns = [col for col in all_cols if col not in drop_columns]
        
        convert_options = pycsv.ConvertOptions(
            column_types=column_types,
            timestamp_parsers=[
                '%Y-%m-%d %H:%M:%S',
                '%Y-%m-%d'
            ],
            include_columns=include_columns,
            null_values=['',' ', 'NA', 'N/A', 'null', 'NULL']
        )

        peek = pycsv.open_csv(csv_path, parse_options=parse_options, convert_options=convert_options)
        base_schema = next(iter(peek)).schema

        #every utf8 col becomes dictionary<int8, utf8>
        final_fields = []
        for field in base_schema:
            if field.type == pa.string():
                final_fields.append(pa.field(field.name, pa.dictionary(pa.int16(), pa.string())))
            else:
                final_fields.append(field)
        final_schema = pa.schema(final_fields)

        reader = pycsv.open_csv(csv_path, parse_options=parse_options, convert_options=convert_options)

        writer = pypq.ParquetWriter(
            output_filename, final_schema,
            compression='zstd',
            use_dictionary=True,
            write_statistics=True
            )
        
        for batch in reader:
            arrays = []
            for i, field in enumerate(final_schema):
                arr = batch.column(i)
                if field.type == pa.dictionary(pa.int16(), pa.string()):
                    arr = arr.cast(field.type)
                arrays.append(arr)
            writer.write_batch(pa.RecordBatch.from_arrays(arrays, schema=final_schema))
        
        writer.close()
        print(f'Successfully wrote Parquet file to: {output_path}')
    
    return output_path

## Early Stopping Class by Jeff Heaton

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

## Converting our Kaggle CSV into a Parquet using PyArrow, Also Converting Strings into Dictionaries

In [47]:
csv_to_parquet_pyarrow("WA_Fn-UseC_-Telco-Customer-Churn.csv", column_types={'customerID': pa.string(), 'TotalCharges': pa.float64()})

df = pd.read_parquet("WA_Fn-UseC_-Telco-Customer-Churn.parquet")

peek_parquet("WA_Fn-UseC_-Telco-Customer-Churn.parquet")

#we have training data but now we want to perform a k fold train tes split to get a better estimate of our model's performance on unseen data. We can use the train_test_split function from sklearn to do this.


Name: 'WA_Fn-UseC_-Telco-Customer-Churn'
Path: 'WA_Fn-UseC_-Telco-Customer-Churn.parquet'
Size: 0.00013 GB
Files: 1
Rows: 7,043
Schema:
    customerID: dictionary<values=string, indices=int16, ordered=0>
    gender: dictionary<values=string, indices=int16, ordered=0>
    SeniorCitizen: int64
    Partner: dictionary<values=string, indices=int16, ordered=0>
    Dependents: dictionary<values=string, indices=int16, ordered=0>
    tenure: int64
    PhoneService: dictionary<values=string, indices=int16, ordered=0>
    MultipleLines: dictionary<values=string, indices=int16, ordered=0>
    InternetService: dictionary<values=string, indices=int16, ordered=0>
    OnlineSecurity: dictionary<values=string, indices=int16, ordered=0>
    OnlineBackup: dictionary<values=string, indices=int16, ordered=0>
    DeviceProtection: dictionary<values=string, indices=int16, ordered=0>
    TechSupport: dictionary<values=string, indices=int16, ordered=0>
    StreamingTV: dictionary<values=string, indices=int16,

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
#missing values in TotalCharges, we can handle them
df['TotalCharges'] = df['TotalCharges'].fillna(0)

y = df['Churn'].values
y = np.where(y == 'Yes', 1, 0)

df = df.drop(columns=['Churn', 'customerID'])
#pd get dummies to convert categorical variables into numerical variables
df = pd.get_dummies(df, drop_first=False)

# Convert to numpy for classification
x = df.values

#standard scaler
scaler = StandardScaler()
x = scaler.fit_transform(x)

In [ ]:
# Assuming your data is in Numpy Arrays. If not, convert them into Numpy Arrays
x = np.array(x)
y = np.array(y)

# Use the nn.Sequential API
model = nn.Sequential(
    nn.Linear(x.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, y.shape[2]),
    nn.Softmax(dim=1),
)
model = torch.compile(model,backend="aot_eager").to(device)

# Cross-validate
kf = StratifiedKFold(5, shuffle=True, random_state=42)

# Defining Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

oos_y = []
oos_pred = []
fold = 0

for train, test in kf.split(x, y):
    fold += 1
    print(f"Fold #{fold}")

    x_train = torch.tensor(x[train], device=device, dtype=torch.float32)
    y_train = torch.tensor(np.argmax(y[train], axis=1),device=device, dtype=torch.long)  # Convert to class indices
    x_test = torch.tensor(x[test],device=device, dtype=torch.float32)
    y_test = torch.tensor(np.argmax(y[test], axis=1),device=device, dtype=torch.long)  # Convert to class indices

    # Training loop
    EPOCHS = 500
    epoch = 0
    done = False
    es = EarlyStopping(restore_best_weights=True)

    while not done and epoch < EPOCHS:
        epoch += 1
        model.train()
        optimizer.zero_grad()
        output = model(x_train)
        loss = criterion(output, y_train)
        loss.backward()
        optimizer.step()

        # Evaluate validation loss
        model.eval()
        with torch.no_grad():
            y_val = model(x_test)
            val_loss = criterion(y_val, y_test)

        if es(model, val_loss):
            done = True

    # Prediction
    with torch.no_grad():
        y_val = model(x_test)
        _, pred = torch.max(y_val, 1)

    oos_y.append(y_test.cpu().numpy())
    oos_pred.append(pred.cpu().numpy())

    print(
        f"Epoch {epoch}/{EPOCHS}, Validation Loss: " f"{val_loss.item()}, {es.status}"
    )

    # Measure this fold's accuracy
    score = metrics.accuracy_score(y_test.cpu().numpy(), pred.cpu().numpy())
    print(f"Fold score (accuracy): {score}")

# Build the oos prediction list and calculate the error.
oos_y = np.concatenate(oos_y)
oos_pred = np.concatenate(oos_pred)

score = metrics.accuracy_score(oos_y, oos_pred)
print(f"Final score (accuracy): {score}")


Train indices: [   0    1    2 ... 7040 7041 7042], Test indices: [   8   14   15 ... 7034 7036 7037]
Train indices: [   0    1    2 ... 7038 7039 7042], Test indices: [  12   26   29 ... 7031 7040 7041]
Train indices: [   1    2    3 ... 7037 7040 7041], Test indices: [   0    6    7 ... 7038 7039 7042]
Train indices: [   0    3    4 ... 7040 7041 7042], Test indices: [   1    2   10 ... 7026 7032 7035]
Train indices: [   0    1    2 ... 7040 7041 7042], Test indices: [   3    4    5 ... 7027 7029 7033]
